In [ ]:
# RUN EACH BLOCK!!!

In [3]:
# 1. Authenticate to Google Earth Engine
import ee

ee.Authenticate()


Successfully saved authorization token.


In [1]:
# 2. Create a Dask cluster

from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=20,                  # number of Python processes
    threads_per_worker=1,         # single-threaded workers
    memory_limit="3GB",           # per worker; can be "auto"
    scheduler_port=0,             # 0 = choose a free port
    dashboard_address=":8787",    # open the dashboard
    processes=True,               # True = separate processes (default)
)
client = Client(cluster)

2025-10-06 15:52:22,635 - distributed.scheduler - WARNING - Worker failed to heartbeat for 619s; attempting restart: <WorkerState 'tcp://127.0.0.1:58425', name: 11, status: running, memory: 0, processing: 0>
2025-10-06 15:52:22,745 - distributed.scheduler - WARNING - Worker failed to heartbeat for 619s; attempting restart: <WorkerState 'tcp://127.0.0.1:58428', name: 13, status: running, memory: 0, processing: 0>
2025-10-06 15:52:22,811 - distributed.scheduler - WARNING - Worker failed to heartbeat for 619s; attempting restart: <WorkerState 'tcp://127.0.0.1:58429', name: 15, status: running, memory: 0, processing: 0>
2025-10-06 15:52:22,834 - distributed.scheduler - WARNING - Worker failed to heartbeat for 619s; attempting restart: <WorkerState 'tcp://127.0.0.1:58430', name: 9, status: running, memory: 0, processing: 0>
2025-10-06 15:52:22,905 - distributed.scheduler - WARNING - Worker failed to heartbeat for 619s; attempting restart: <WorkerState 'tcp://127.0.0.1:58431', name: 8, statu

In [2]:
# 3. Initialize Dask workers with Google Earth Engine
import ee
from distributed import WorkerPlugin

class EEPlugin(WorkerPlugin):
    def __init__(self):
        pass
    def setup(self, worker):
        self.worker = worker
        try:
            # Assume credentials already exist at default location
            ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')
        except ee.EEException as e:
            raise RuntimeError("Earth Engine initialization failed. "
                            "Run ee.Authenticate() once on the client before starting the cluster.")

ee_plugin = EEPlugin()
client.register_plugin(ee_plugin)

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


{'tcp://127.0.0.1:58425': {'status': 'OK'},
 'tcp://127.0.0.1:58428': {'status': 'OK'},
 'tcp://127.0.0.1:58429': {'status': 'OK'},
 'tcp://127.0.0.1:58430': {'status': 'OK'},
 'tcp://127.0.0.1:58431': {'status': 'OK'},
 'tcp://127.0.0.1:58440': {'status': 'OK'},
 'tcp://127.0.0.1:58441': {'status': 'OK'},
 'tcp://127.0.0.1:58446': {'status': 'OK'},
 'tcp://127.0.0.1:58447': {'status': 'OK'},
 'tcp://127.0.0.1:58448': {'status': 'OK'},
 'tcp://127.0.0.1:58455': {'status': 'OK'},
 'tcp://127.0.0.1:58456': {'status': 'OK'},
 'tcp://127.0.0.1:58457': {'status': 'OK'},
 'tcp://127.0.0.1:58458': {'status': 'OK'},
 'tcp://127.0.0.1:58459': {'status': 'OK'},
 'tcp://127.0.0.1:58470': {'status': 'OK'},
 'tcp://127.0.0.1:58473': {'status': 'OK'},
 'tcp://127.0.0.1:58474': {'status': 'OK'},
 'tcp://127.0.0.1:58479': {'status': 'OK'},
 'tcp://127.0.0.1:58482': {'status': 'OK'}}

In [3]:
# 4. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

In [4]:
# 5. Load Earth Engine data into an xarray dataset via xee

import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=US_Boundaries.geometry(), crs='EPSG:5070', scale=30, chunks='auto')

In [5]:
# 6. Compute NDVI using the xarray
import numpy as np

def add_ndvi(ds: xr.Dataset) -> xr.Dataset:
    # Bands: B5 = NIR, B4 = Red
    nir = ds["SR_B5"].astype("float32")
    red = ds["SR_B4"].astype("float32")

    # Compute NDVI, guard against divide-by-zero
    denom = nir + red
    ndvi = xr.where(denom != 0, (nir - red) / denom, np.nan).astype("float32")

    # Add attributes for clarity
    ndvi = ndvi.assign_attrs(
        {
            "long_name": "Normalized Difference Vegetation Index",
            "standard_name": "NDVI",
            "description": "NDVI = (SR_B5 - SR_B4) / (SR_B5 + SR_B4)",
            "units": "1",
            "source_bands": "SR_B5 (NIR), SR_B4 (Red)",
        }
    )

    # Mutate dataset directly
    return xr.Dataset({"ndvi": ndvi})

In [ ]:
# --- Replace your steps 6–7 with this ---

import numpy as np
import xarray as xr

# Pull bands as dask-backed DataArrays (already chunked from open_dataset)
nir = ds["SR_B5"].astype("float32")
red = ds["SR_B4"].astype("float32")

# Element-wise NDVI with divide-by-zero guard
denom = nir + red
ndvi = xr.where(denom != 0, (nir - red) / denom, np.nan).astype("float32")

# Nice metadata and name
ndvi = ndvi.rename("ndvi").assign_attrs(
    {
        "long_name": "Normalized Difference Vegetation Index",
        "standard_name": "NDVI",
        "description": "NDVI = (SR_B5 - SR_B4) / (SR_B5 + SR_B4)",
        "units": "1",
        "source_bands": "SR_B5 (NIR), SR_B4 (Red)",
    }
)

# If you prefer a Dataset (like your original)
results = ndvi.to_dataset()

# Submit to the cluster. Use .persist() to keep results on workers if you’ll reuse them;
# use .compute() if you need the concrete in-memory result immediately.
results = results.persist()   # or: results = results.compute()


In [8]:
client.close()

In [ ]:
# 7. Run the computation, which should replicate the warning.
results = xr.map_blocks(func=add_ndvi, obj=ds)
results.compute()

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\distributed\client.py:3362: UserWarning: Sending large graph of size 368.26 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [4]:
# 1) Open with bigger chunks and only needed bands
import xarray as xr

ds = xr.open_dataset(
    ic,
    engine="ee",
    geometry=US_Boundaries.geometry(),
    crs="EPSG:5070",
    scale=5000,
    chunks={"time": 48, "x": 512, "y": 256},
)[["SR_B4", "SR_B5"]]

# 2) Optional: checkpoint to truncate deep upstream graphs
ds = ds.persist()  # or use to_zarr/open_zarr if memory is tight

# 3) Elementwise NDVI with apply_ufunc (compact graph)
def _ndvi(nir, red):
    denom = nir + red
    return xr.where(denom != 0, (nir - red) / denom, np.nan).astype("float32")

ndvi = xr.apply_ufunc(
    _ndvi, ds["SR_B5"], ds["SR_B4"],
    dask="parallelized", output_dtypes=[np.float32]
).assign_attrs(
    long_name="Normalized Difference Vegetation Index",
    standard_name="NDVI",
    description="NDVI = (SR_B5 - SR_B4) / (SR_B5 + SR_B4)",
    units="1", source_bands="SR_B5 (NIR), SR_B4 (Red)"
)

# 4) Compute or write out
ndvi.compute()    # or ndvi.to_zarr("ndvi.zarr", mode="w")


EEException: Too Many Requests: Request was rejected because the request rate or concurrency limit was exceeded.